In [55]:
import torch
import torch.nn as nn

import pandas as pd

from src.preprocessing import build_preprocessor

import numpy as np
from sklearn.model_selection import StratifiedKFold

In [ ]:
#%cd ..

%load_ext autoreload
%autoreload 2

c:\Users\coffe\Desktop\titanic-ml\titanic-ml


In [4]:
SEED = 42

In [64]:
class TitanicMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            nn.Linear(32, 32, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            
            nn.Linear(32, 1)
        )
        
    def forward(self, x):
        return self.net(x)

In [6]:
from sklearn.model_selection import train_test_split

train = pd.read_csv("data/train.csv")

X = train.drop(columns=['Survived'])
y = train['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

In [10]:
preprocessor = build_preprocessor('standard')

X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_test)

X_train_processed.shape, X_val_processed.shape

((712, 13), (179, 13))

In [13]:
X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_processed, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
y_val_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

torch.Size([712, 13])
torch.Size([712, 1])


In [87]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

In [88]:
x_batch, y_batch = next(iter(train_loader))

print(x_batch.shape)
print(y_batch.shape)

torch.Size([64, 13])
torch.Size([64, 1])


In [53]:
torch.manual_seed(SEED)

model = TitanicMLP(input_dim=13)

loss_fn = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
EPOCHS = 100

best_val_accuracy = 0.0
best_epoch = 0

for epoch in range(EPOCHS):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0
    
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        
        logits = model(x_batch)
        loss = loss_fn(logits, y_batch)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        predictions = (torch.sigmoid(logits) >= 0.5).float()
        
        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)
    
    mean_train_loss = running_loss / len(train_loader)
    accuracy = correct / total if total > 0 else 0

    model.eval()
    
    val_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            
            val_loss += loss.item()
            
            predictions = (torch.sigmoid(logits) >= 0.5).float()
            
            correct += (predictions == y_batch).sum().item()
            total += y_batch.size(0)
            
    mean_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total
    
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_epoch = epoch + 1
    
    print(f"Epoch {epoch+1}/{EPOCHS} - Train Loss: {mean_train_loss:.4f}, Train Acc: {accuracy:.4f}, Val Loss: {mean_val_loss:.4f}, Val Acc: {val_accuracy:.4f}")
    
print(
    f"Best Val Acc: {best_val_accuracy:.4f} "
    f"at epoch {best_epoch}"
)

In [89]:
dropout = 0.0

class TitanicMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, 16, bias=False),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(16, 16, bias=False),
            nn.BatchNorm1d(16),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            nn.Linear(16, 1)
        )
        
    def forward(self, x):
        return self.net(x)

In [93]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

fold_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X, y), start=1):
    print(f'\nFold {fold}')
    
    # 1. Split
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]
    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]
    
    # 2. Preprocessing
    preprocessor = build_preprocessor('standard')
    
    X_train_processed = preprocessor.fit_transform(X_train)
    X_val_processed = preprocessor.transform(X_val)
    
    # 3. Tensors
    X_train_tensor = torch.tensor(X_train_processed, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val_processed, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
    y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
    
    # 4. Dataloaders
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=32, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=32, shuffle=False)
    
    # 5. Model, Loss, Optimizer
    torch.manual_seed(SEED)
    
    model = TitanicMLP(input_dim=13)
    
    loss_fn =  nn.BCEWithLogitsLoss()
    
    EPOCHS = 100
    
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    
    best_val_accuracy = 0.0
    
    for epoch in range(EPOCHS):
        model.train()
        
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            
            logits = model(x_batch)
            loss = loss_fn(logits, y_batch)
            
            loss.backward()
            optimizer.step()
            
        model.eval()
        
        correct = 0
        total = 0
        
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                logits = model(x_batch)
                
                predictions = (torch.sigmoid(logits) >= 0.5).float()
                
                correct += (predictions == y_batch).sum().item()
                total += y_batch.size(0)
                
        val_accuracy = correct / total
        
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
    
        
            
    print(f"Best Val Acc: {best_val_accuracy:.4f}")
    
    fold_scores.append(best_val_accuracy)
    
fold_scores = np.array(fold_scores)

print("\nDNN 5-fold CV")
print("Scores:", fold_scores)
print("Mean:", fold_scores.mean())
print("Std:", fold_scores.std())


Fold 1
Best Val Acc: 0.8659

Fold 2
Best Val Acc: 0.8596

Fold 3
Best Val Acc: 0.8258

Fold 4
Best Val Acc: 0.8652

Fold 5
Best Val Acc: 0.8315

DNN 5-fold CV
Scores: [0.86592179 0.85955056 0.8258427  0.86516854 0.83146067]
Mean: 0.849588851923922
Std: 0.017327653245296698
